# 01 Preprocessing

Preprocessing-only notebook for loading cleaned ASHRAE data and building engineered features.


## Scope
- Load cleaned train/weather/building metadata tables
- Merge and validate columns
- Build preprocessing features (time, weather, metadata, lag/rolling, encodings)


## 1. Environment Setup and Data Loading

In [1]:
import os, random, math, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean, cosine as cosine_dist
from scipy.signal import savgol_filter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1a. Generate Cleaned Input Files
Run these once (or whenever raw data/preprocessing params change) before loading data below.


In [2]:
import subprocess, sys

RUN_PREPROCESS_SCRIPTS = True  

if RUN_PREPROCESS_SCRIPTS:
    cmd = [sys.executable, "ashrae/preprocess_isamu_matt.py", "--input-dir", "ashrae", "--output-dir", "ashrae"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped: preprocess_isamu_matt.py")


Running: c:\Users\tamar\AppData\Local\Programs\Python\Python314\python.exe ashrae/preprocess_isamu_matt.py --input-dir ashrae --output-dir ashrae


In [3]:
import subprocess, sys

RUN_MEAN_FILTER_SCRIPT = True  

if RUN_MEAN_FILTER_SCRIPT:
    cmd = [sys.executable, "ashrae/make_building_mean_y_ge_1.py"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped: make_building_mean_y_ge_1.py")


Running: c:\Users\tamar\AppData\Local\Programs\Python\Python314\python.exe ashrae/make_building_mean_y_ge_1.py


In [4]:
# # ============================================================
# # Geographic and Building-Type Heterogeneity Analysis
# # ============================================================
 
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
 
# # ------------------------------------------------------------
# # BASIC DATASET OVERVIEW
# # ------------------------------------------------------------
 
# print("Total rows:", len(df))
# print("Total buildings:", df["building_id"].nunique())
# print("Total sites:", df["site_id"].nunique())
# print("Building types:", df["primary_use"].nunique())
 
# # ------------------------------------------------------------
# # SITE-LEVEL HETEROGENEITY
# # ------------------------------------------------------------
 
# site_summary = (
#     df.groupby("site_id")
#       .agg(
#           n_buildings=("building_id", "nunique"),
#           n_primary_uses=("primary_use", "nunique"),
#           avg_meter_reading=("meter_reading", "mean"),
#           std_meter_reading=("meter_reading", "std"),
#           avg_air_temp=("air_temperature", "mean"),
#           std_air_temp=("air_temperature", "std"),
#           avg_square_feet=("square_feet", "mean")
#       )
#       .reset_index()
#       .sort_values("n_buildings", ascending=False)
# )
 
# print("\n================ SITE-LEVEL DIVERSITY ================\n")
# display(site_summary)
 
# # ------------------------------------------------------------
# # BUILDING-TYPE DIVERSITY
# # ------------------------------------------------------------
 
# building_type_summary = (
#     df.groupby(["site_id", "primary_use"])
#       .agg(
#           n_buildings=("building_id", "nunique"),
#           avg_meter_reading=("meter_reading", "mean"),
#           avg_air_temp=("air_temperature", "mean"),
#           avg_square_feet=("square_feet", "mean")
#       )
#       .reset_index()
#       .sort_values(["site_id", "n_buildings"], ascending=[True, False])
# )
 
# print("\n================ BUILDING-TYPE DIVERSITY ================\n")
# display(building_type_summary)
 
# # ------------------------------------------------------------
# # NON-IID EVIDENCE
# # Average consumption differs significantly across sites
# # ------------------------------------------------------------
 
# site_energy = (
#     df.groupby("site_id")["meter_reading"]
#       .mean()
#       .sort_values(ascending=False)
# )
 
# print("\n================ AVERAGE ENERGY CONSUMPTION PER SITE ================\n")
# print(site_energy)
 
# # ------------------------------------------------------------
# # CLIMATE DIVERSITY
# # ------------------------------------------------------------
 
# weather_diversity = (
#     df.groupby("site_id")
#       .agg(
#           mean_temp=("air_temperature", "mean"),
#           std_temp=("air_temperature", "std"),
#           min_temp=("air_temperature", "min"),
#           max_temp=("air_temperature", "max"),
#           mean_dew_temp=("dew_temperature", "mean"),
#           mean_wind_speed=("wind_speed", "mean")
#       )
#       .reset_index()
# )
 
# print("\n================ CLIMATE DIVERSITY ================\n")
# display(weather_diversity)
 
# # ------------------------------------------------------------
# # VISUALIZATION 1:
# # Building-Type Distribution Across Sites
# # ------------------------------------------------------------
 
# pivot_buildings = (
#     df.drop_duplicates("building_id")
#       .pivot_table(
#           index="site_id",
#           columns="primary_use",
#           values="building_id",
#           aggfunc="count",
#           fill_value=0
#       )
# )
 
# pivot_buildings.plot(
#     kind="bar",
#     stacked=True,
#     figsize=(16, 7)
# )
 
# plt.title("Building-Type Diversity Across Geographic Sites")
# plt.xlabel("Site ID")
# plt.ylabel("Number of Buildings")
# plt.legend(
#     bbox_to_anchor=(1.02, 1),
#     loc="upper left",
#     fontsize=8
# )
# plt.tight_layout()
# plt.show()
 
# # ------------------------------------------------------------
# # VISUALIZATION 2:
# # Average Energy Consumption by Site
# # ------------------------------------------------------------
 
# plt.figure(figsize=(10, 5))
 
# site_energy.plot(kind="bar")
 
# plt.title("Average Energy Consumption per Geographic Site")
# plt.xlabel("Site ID")
# plt.ylabel("Average Meter Reading")
# plt.tight_layout()
# plt.show()
 
# # ------------------------------------------------------------
# # VISUALIZATION 3:
# # Climate Heterogeneity Across Sites
# # ------------------------------------------------------------
 
# plt.figure(figsize=(10, 5))
 
# plt.bar(
#     weather_diversity["site_id"].astype(str),
#     weather_diversity["mean_temp"]
# )
 
# plt.title("Average Air Temperature Across Sites")
# plt.xlabel("Site ID")
# plt.ylabel("Mean Air Temperature")
# plt.tight_layout()
# plt.show()
 
# # ------------------------------------------------------------
# # OPTIONAL:
# # Correlation Matrix for Site-Level Characteristics
# # ------------------------------------------------------------
 
# numeric_site_features = site_summary.select_dtypes(include=[np.number])
 
# correlation_matrix = numeric_site_features.corr()
 
# print("\n================ SITE-LEVEL CORRELATION MATRIX ================\n")
# display(correlation_matrix)
 
# # ------------------------------------------------------------
# # FINAL INTERPRETATION
# # ------------------------------------------------------------
 
# print("""
# Interpretation:
# - Different sites exhibit substantially different average
#   energy consumption and climate statistics.
# - Building-type distributions vary across sites.
# - These differences confirm strong non-IID behavior
#   across federated clients.
# - Geographic clustering therefore introduces realistic
#   heterogeneity into the federated learning setup,
#   motivating similarity-aware and transfer-based methods.
# """)

## Result
df contains the preprocessed dataset ready for downstream training.
